In [1]:
!pip install torch  pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 51.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
import pandas as pd
from datetime import datetime
import uuid
from sklearn.metrics import r2_score
import ast
import itertools, os, time, json
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# ---------------------- SEED ----------------------
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
# ---------------------- DRIVE & DATA ----------------------
from google.colab import drive
drive.mount('/content/drive')

BASE_SAVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"

file_path = f'{BASE_SAVE_ROOT}/Final_cps_timeseries_features.csv'
data = pd.read_csv(file_path)
TARGETS = ["Buffer_M1_M2", "Buffer_M2_M3"]
if True:
    !python "/content/drive/MyDrive/Colab Notebooks/preprocessing.py" \
    --csv "/content/drive/MyDrive/Colab Notebooks/Final_cps_timeseries_features.csv" \
    --pipelines BASE Y_LAGS \
    --y_lags 1 \
    --scale_x --scale_y \
    --window_sizes 32 \
    --outdir "/content/drive/MyDrive/Colab Notebooks/artifacts_preprocessed"
# ---------------------- QUANTUM CIRCUIT ----------------------
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]
# ---------------------- Q LAYER FACTORY ----------------------
def make_q_layer(q_depth, n_qubits=n_qubits):
    weight_shapes = {"weights": (q_depth, n_qubits)}
    return qml.qnn.TorchLayer(qnode, weight_shapes)
# ---------------------- HYBRID MODELS ----------------------
class HybridQNNRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, q_layer, num_layers=1, nonlinearity='tanh'):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layers,nonlinearity=nonlinearity, batch_first=True)
        self.to_q = nn.Linear(hidden_size, n_qubits)
        self.fc = nn.Linear(n_qubits, output_size)
        self.q_layer = q_layer
    def forward(self, x):
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        features = self.to_q(last)
        qnn_out = self.q_layer(features)
        return self.fc(qnn_out)
class HybridQNNLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, q_layer, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.to_q = nn.Linear(hidden_size, n_qubits)
        self.fc = nn.Linear(n_qubits, output_size)
        self.q_layer = q_layer
    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        features = self.to_q(last)
        qnn_out = self.q_layer(features)
        return self.fc(qnn_out)
# ---------------------- PARAM COUNT ----------------------
def count_nn_params(model):
    return sum(p.numel() for n,p in model.named_parameters() if "q_layer" not in n)
def count_qnn_params(model):
    q_params = 0
    for m in model.modules():
        if isinstance(m, qml.qnn.TorchLayer):
            for p in m.parameters():
                q_params += p.numel()
    return q_params
# ---------------------- METRICS ----------------------
def compute_seq_metrics_exact(preds: np.ndarray, targs: np.ndarray, target_names=None):
    assert preds.shape == targs.shape
    N, T = targs.shape
    if not target_names or len(target_names) != T:
        target_names = [f"target_{i}" for i in range(T)]
    err = targs - preds
    mse_t  = np.mean(err**2, axis=0)
    mae_t  = np.mean(np.abs(err), axis=0)
    rmse_t = np.sqrt(mse_t)
    ss_res = np.sum(err**2, axis=0)
    ss_tot = np.sum((targs - targs.mean(axis=0))**2, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        r2_t = 1.0 - np.where(ss_tot > 0, ss_res / ss_tot, 0.0)
    mse  = float(np.mean(err**2))
    mae  = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))
    r2   = float(r2_score(targs, preds))
    weights = ss_tot / (ss_tot.sum() + 1e-12)
    r2_macro  = float(np.sum(r2_t * weights))
    metrics = {"Test_MSE": mse, "Test_MAE": mae, "Test_RMSE": rmse, "Test_R2": r2,"Test_R2_macro": r2_macro, "PerTarget_MSE": mse_t.tolist(),
               "PerTarget_MAE": mae_t.tolist(),"PerTarget_RMSE": rmse_t.tolist(), "PerTarget_R2": r2_t.tolist()}
    per_target_rows = [{"target": target_names[i],"MSE": float(mse_t[i]),"MAE": float(mae_t[i]),"RMSE": float(rmse_t[i]),
                        "R2": float(r2_t[i]),"var_weight": float(weights[i])} for i in range(T)]
    return metrics, per_target_rows
# ---------------------- PREDICT + INVERSE ----------------------
def predict_and_invert(model, x_tensor, y_tensor, batch_size, inv_y):
    model.eval()
    preds, targs = [], []
    with torch.no_grad():
        for i in range(0, len(x_tensor), batch_size):
            xb = x_tensor[i:i+batch_size]
            yb = y_tensor[i:i+batch_size]
            out = model(xb)
            preds.append(out.detach().cpu().numpy())
            targs.append(yb.detach().cpu().numpy())
    preds = np.vstack(preds)
    targs = np.vstack(targs)
    preds = inv_y(preds)
    targs = inv_y(targs)
    return preds, targs
# ---------------------- TRAIN FINAL ----------------------
def plot_predictions_subplots(preds, targs, target_names, out_path, figsize=(14, 3), auto_zoom=True, zoom_window_size=100, top_k=3):
    """ Plot predictions vs. targets for multiple time series targets.
    Features: - Consistent y-axis across targets
              - Full-range plot with shaded high-error regions (red zones)
              - Optional zoomed-in plots of top-error regions
              - High-res output for publications (PNG + PDF)
    """
    import matplotlib.patches as patches  # Ensure this is imported

    n_targets = len(target_names)
    # Determine global y-limits
    y_min = min(targs.min(), preds.min())
    y_max = max(targs.max(), preds.max())

    # Compute top-error windows if auto_zoom is enabled
    zoom_windows = None
    if auto_zoom:
        errors = np.abs(targs - preds).mean(axis=1)
        top_indices = errors.argsort()[-top_k:][::-1]
        zoom_windows = [(max(0, idx - zoom_window_size//2),
                        min(len(preds), idx + zoom_window_size//2))
                        for idx in top_indices]

    # --- Full-range plot ---
    fig, axes = plt.subplots(n_targets, 1, figsize=(figsize[0], figsize[1]*n_targets), dpi=300)
    if n_targets == 1:
        axes = [axes]

    for i, name in enumerate(target_names):
        axes[i].plot(targs[:, i], label="Actual", color='tab:blue', linewidth=1.5)
        axes[i].plot(preds[:, i], label="Pred", color='tab:orange', alpha=0.8, linewidth=1.5)
        axes[i].set_ylabel(name)
        axes[i].set_xlabel("Time Step")
        axes[i].set_ylim(y_min, y_max)
        axes[i].grid(True, linestyle="--", alpha=0.5)
        axes[i].legend(loc='upper left', fontsize=10)

        # ADD BACK: Shade high-error regions (red zones) in full-range plot
        if zoom_windows:
            for start, end in zoom_windows:
                axes[i].add_patch(patches.Rectangle(
                    (start, y_min),           # x, y position
                    end - start,              # width
                    y_max - y_min,            # height
                    color='red',
                    alpha=0.1,                # transparent red
                    zorder=0                  # behind the lines
                ))

    plt.tight_layout()
    # Save PNG + PDF
    plt.savefig(out_path, dpi=300)
    plt.savefig(out_path.replace(".png", ".pdf"), dpi=300, format='pdf')
    plt.close()

    # --- Zoomed-in plots ---
    if zoom_windows:
        for start, end in zoom_windows:
            fig, axes = plt.subplots(n_targets, 1, figsize=(figsize[0], figsize[1]*n_targets), dpi=300)
            if n_targets == 1:
                axes = [axes]

            for i, name in enumerate(target_names):
                axes[i].plot(targs[start:end, i], label="Actual", color='tab:blue', linewidth=1.5)
                axes[i].plot(preds[start:end, i], label="Pred", color='tab:orange', alpha=0.8, linewidth=1.5)
                axes[i].set_ylabel(name)
                axes[i].set_xlabel("Time Step")
                axes[i].set_ylim(y_min, y_max)
                axes[i].grid(True, linestyle="--", alpha=0.5)
                axes[i].legend(loc='upper left', fontsize=10)

            plt.tight_layout()
            zoom_png = out_path.replace(".png", f"_zoom_{start}_{end}.png")
            zoom_pdf = out_path.replace(".png", f"_zoom_{start}_{end}.pdf")
            plt.savefig(zoom_png, dpi=300)
            plt.savefig(zoom_pdf, dpi=300, format='pdf')
            plt.close()
def train_final_and_plot(model_class,x_train, y_train,x_val, y_val,x_test, y_test,inv_y,hidden_size, num_layers,lr, q_depth,batch_size, epochs,target_names, out_dir,patience=5):
    """ Train a hybrid RNN/LSTM + QNN model with early stopping on validation loss.
    Returns: model, train_curve, val_curve, predictions, targets, metrics"""
    # --- Make QNN layer ---
    q_layer_instance = make_q_layer(q_depth)
    model = model_class(input_size=x_train.shape[-1],
                        hidden_size=hidden_size,
                        output_size=y_train.shape[-1],
                        num_layers=num_layers,
                        q_layer=q_layer_instance)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_curve = []
    val_curve = []
    best_val_loss = float('inf')
    converge_epoch = 0
    no_improve = 0
    start_time = time.time()

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        epoch_train_loss = 0.0
        for i in range(0, len(x_train), batch_size):
            xb = x_train[i:i+batch_size]
            yb = y_train[i:i+batch_size]
            out = model(xb)
            loss = criterion(out, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item() * len(xb)
        epoch_train_loss /= len(x_train)
        train_curve.append(epoch_train_loss)
        # --- Validation ---
        model.eval()
        with torch.no_grad():
            val_out = model(x_val)
            val_loss = criterion(val_out, y_val).item()
            val_curve.append(val_loss)
        # --- Early stopping on validation ---
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            converge_epoch = epoch + 1
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    stop_epoch = epoch + 1
    total_time = time.time() - start_time
    # --- Predictions & metrics ---
    preds, targs = predict_and_invert(model, x_test, y_test, batch_size, inv_y)
    metrics, per_target_rows = compute_seq_metrics_exact(preds, targs, target_names)
    metrics.update({"train_time_sec": total_time,
                    "converge_epoch": converge_epoch,
                    "stop_epoch": stop_epoch})
    # --- Param counts ---
    nn_params = count_nn_params(model)
    qnn_params = count_qnn_params(model)
    print(f"NN parameters: {nn_params}, QNN parameters: {qnn_params}")
    print("Metrics:", metrics)

    # --- Save predictions plot ---
    n_targets = len(target_names)
    fig, axes = plt.subplots(n_targets, 1, figsize=(14, 3*n_targets), dpi=300)
    if n_targets == 1:
        axes = [axes]
    for i, t in enumerate(target_names):
        axes[i].plot(targs[:,i], label=f"{t} actual", color='tab:blue', linewidth=1.5)
        axes[i].plot(preds[:,i], label=f"{t} pred", color='tab:orange', alpha=0.8, linewidth=1.5)
        axes[i].set_ylabel(t, fontsize=12)
        axes[i].set_xlabel("Time Step", fontsize=12)
        axes[i].grid(True, linestyle="--", alpha=0.5)
        axes[i].legend(loc='upper left', fontsize=10)
        axes[i].tick_params(labelsize=10)
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)

    pred_png = os.path.join(out_dir, f"{model_class.__name__}_predictions.png")
    pred_pdf = os.path.join(out_dir, f"{model_class.__name__}_predictions.pdf")
    plt.savefig(pred_png, dpi=300)
    plt.savefig(pred_pdf, dpi=300, format='pdf')
    plt.close()

    # --- Save training + validation curve plot ---
    plt.figure(figsize=(7, 5), dpi=300)
    plt.plot(train_curve, label="Train Loss", color='tab:blue', linewidth=2, marker='o', markersize=4)
    plt.plot(val_curve, label="Validation Loss", color='tab:orange', linewidth=2, linestyle='--', marker='s', markersize=4)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("MSE Loss", fontsize=12)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=10)
    plt.tight_layout()

    curve_png = os.path.join(out_dir, f"{model_class.__name__}_train_val_curve.png")
    curve_pdf = os.path.join(out_dir, f"{model_class.__name__}_train_val_curve.pdf")
    plt.savefig(curve_png, dpi=300)
    plt.savefig(curve_pdf, dpi=300, format='pdf')  # FIXED: use correct path
    plt.close()

    return model, train_curve, val_curve, preds, targs, metrics
# ---------------------- GRID SEARCH ----------------------
def grid_products(param_grid):
    keys = param_grid.keys()
    vals = param_grid.values()
    for prod in itertools.product(*vals):
        yield dict(zip(keys, prod))
# ---------------------- TIME SERIES SPLITS ----------------------
def get_time_series_splits(X, n_splits=3, val_frac=0.15):
    n_samples = len(X)
    test_size = int(np.ceil(n_samples * val_frac))
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    return list(tscv.split(X))
# ---------------------- RUN EXPERIMENT (WITH FOLDS) ----------------------
def run_experiment(params, x_data, y_data, run_dir):
    """Run one grid search configuration using rolling time series splits.
    Aggregates metrics across folds."""
    model_class = params['model_class']
    fold_metrics = []
    fold_predictions_all = []  # NEW: store per-fold predictions for this run
    TIME_FOLDS = get_time_series_splits(x_data, n_splits=3, val_frac=0.15)

    for fold_idx, (train_idx, val_idx) in enumerate(TIME_FOLDS, 1):
        print(f"Params: {params} | Fold {fold_idx}/{len(TIME_FOLDS)}")

        x_train_fold, y_train_fold = x_data[train_idx], y_data[train_idx]
        x_val_fold, y_val_fold     = x_data[val_idx], y_data[val_idx]
        # --- Train model on this fold ---
        model, train_curve,val_curve, y_pred, y_true, metrics = train_final_and_plot(model_class=model_class,x_train=x_train_fold,y_train=y_train_fold,x_val=x_val_fold,
                                                                                     y_val=y_val_fold,x_test=x_val_fold,y_test=y_val_fold,inv_y=inv_y,
                                                                                     hidden_size=params['hidden_size'],num_layers=params['num_layers'],
                                                                                     lr=params['lr'],q_depth=params['q_depth'],batch_size=params['batch_size'],
                                                                                     epochs=params['epochs'],target_names=TARGETS,out_dir=CHECKPOINT_DIR,patience=5)
        # --- Save fold-wise predictions in unique folder ---
        fold_dir = os.path.join(run_dir, f"fold_{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)

        val_seq_idx = val_idx  # validation indices in original dataset
        np.savez(os.path.join(fold_dir, "predictions.npz"),
                 preds=y_pred, targs=y_true, val_seq_idx=val_seq_idx)

        fold_metrics.append(metrics)
        fold_predictions_all.append({"fold": fold_idx,"preds": y_pred,"targs": y_true,"val_seq_idx": val_seq_idx})
    # --- Aggregate metrics across folds ---
    avg_metrics = {}
    metric_keys = fold_metrics[0].keys()
    for key in metric_keys:
        # Only aggregate numeric metrics
        if isinstance(fold_metrics[0][key], (int, float)):
            avg_metrics[key] = np.mean([m[key] for m in fold_metrics])
        else:
            avg_metrics[key] = fold_metrics[-1][key]  # just take last fold for non-numeric
    # --- Combine parameters + averaged metrics ---
    result_row = params.copy()
    result_row.update(avg_metrics)
     # Save all fold predictions for this run
    np.savez(os.path.join(run_dir, f"all_fold_predictions.npz"),
             fold_predictions=fold_predictions_all)
    return result_row
# ---------------------- DATA PICK ----------------------
BASE_DIR = f"{BASE_SAVE_ROOT}/artifacts_preprocessed"
PIPE, W = "Y_LAGS", 32
ROOT = f"{BASE_DIR}/{PIPE}/win{W}"

X_tr = np.load(f"{ROOT}/Xseq_train.npy")
Y_tr = np.load(f"{ROOT}/Yseq_train.npy")
X_va = np.load(f"{ROOT}/Xseq_val.npy")
Y_va = np.load(f"{ROOT}/Yseq_val.npy")
X_te = np.load(f"{ROOT}/Xseq_test.npy")
Y_te = np.load(f"{ROOT}/Yseq_test.npy")
x_tr = torch.from_numpy(X_tr).float()
y_tr = torch.from_numpy(Y_tr).float()
x_val = torch.from_numpy(X_va).float()
y_val = torch.from_numpy(Y_va).float()
x_test = torch.from_numpy(X_te).float()
y_test = torch.from_numpy(Y_te).float()

ysc_path = f"{BASE_DIR}/{PIPE}/y_scaler_state.npz"
if os.path.exists(ysc_path):
    st = np.load(ysc_path)
    y_mean = st["mean"]
    y_std  = st["std"]
    inv_y = lambda a: a*y_std + y_mean
else:
    inv_y = lambda a: a
# ---------------------- EXPERIMENT FOLDER ----------------------
EXPERIMENT_TAG = datetime.now().strftime("%Y%m%d-%H%M%S") + "_" + uuid.uuid4().hex[:6]
EXP_DIR = os.path.join(BASE_SAVE_ROOT, f"tuning_{EXPERIMENT_TAG}")
CHECKPOINT_DIR = os.path.join(EXP_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# ---------------------- GRID SEARCH RUN ----------------------
param_grid = {"model_class": [HybridQNNLSTM, HybridQNNRNN],"hidden_size": [4,8],"num_layers":[1],"lr":[1e-3],"batch_size":[32],"q_depth":[2,3],"epochs":[80]}
results = []
total_runs = np.prod([len(v) for v in param_grid.values()])

for idx, p in enumerate(grid_products(param_grid), 1):
  # Unique folder for this parameter combination
    run_dir = os.path.join(CHECKPOINT_DIR, f"run_{idx}_{p['model_class'].__name__}_hs{p['hidden_size']}_qd{p['q_depth']}")
    os.makedirs(run_dir, exist_ok=True)
    print(f"\n=== Running grid search {idx}/{total_runs} ===")
    print(f"Current params: {p}")
    start_time = time.time()
    result = run_experiment(p, x_tr, y_tr,run_dir)
    elapsed = time.time() - start_time
    print(f"Finished run {idx}/{total_runs} in {elapsed:.2f}s")
    print(f"Averaged metrics for this run: Test_R2_macro={result['Test_R2_macro']:.4f}, Test_MSE={result['Test_MSE']:.4f}")
    results.append(result)
df = pd.DataFrame(results).sort_values(by=["Test_R2_macro","Test_MSE"], ascending=[False, True])
df.to_csv(os.path.join(EXP_DIR,"tuning_results.csv"), index=False)
print(f"\nAll tuning results saved to {os.path.join(EXP_DIR, 'tuning_results.csv')}")
# ---------------------- FINAL TRAINING FOR BOTH MODELS ----------------------
final_models = {}
for model_class in [HybridQNNRNN, HybridQNNLSTM]:
    # Filter DataFrame for this model class
    df_model = df[df["model_class"] == model_class].sort_values(by=["Test_R2_macro", "Test_MSE"], ascending=[False, True])
    if df_model.empty:
        print(f"No results found for {model_class.__name__}. Skipping final training.")
        continue
    best_row = df_model.iloc[0]  # best hyperparameters
    # Concatenate train + validation if you want final training on full data
    x_train_full = torch.cat([x_tr, x_val], dim=0)
    y_train_full = torch.cat([y_tr, y_val], dim=0)

    final_out = os.path.join(EXP_DIR, f"final_model_{model_class.__name__}")
    os.makedirs(final_out, exist_ok=True)
    print(f"\n=== TRAIN FINAL MODEL: {model_class.__name__} ===")
    # Train final model using best hyperparameters
    final_model, train_curve, val_curve, y_pred, y_true, metrics = train_final_and_plot(model_class=model_class,x_train=x_tr,y_train=y_tr,
                                                                                        x_val=x_val,y_val=y_val,x_test=x_test,y_test=y_test,
                                                                                        inv_y=inv_y, hidden_size=int(best_row["hidden_size"]),
                                                                                        num_layers=int(best_row["num_layers"]),lr=float(best_row["lr"]),
                                                                                        q_depth=int(best_row["q_depth"]),batch_size=int(best_row["batch_size"]),
                                                                                        epochs=int(best_row["epochs"]),target_names=TARGETS,out_dir=final_out,patience=5)
    # Plot predictions
    plot_predictions_subplots( preds=y_pred,targs=y_true,target_names=TARGETS,out_path=os.path.join(final_out, f"{model_class.__name__}_predictions.png"),figsize=(14, 3))
    # Convert metrics to a DataFrame and aggregate to a single row (mean over targets)
    metrics_df = pd.DataFrame(metrics)
    metrics_summary = pd.DataFrame([metrics_df.mean()])  # single row
    # Combine metrics summary with best hyperparameters
    best_params_df = best_row.to_frame().T.reset_index(drop=True)
    final_df = pd.concat([metrics_summary, best_params_df], axis=1)
    # Save combined CSV
    metrics_csv_path = os.path.join(final_out, "final_model_metrics.csv")
    final_df.to_csv(metrics_csv_path, index=False)
    print(f"Saved final metrics + best hyperparameters to: {metrics_csv_path}")
    # Store final model info
    final_models[model_class.__name__] = {"model": final_model,"train_curve": train_curve,"y_pred": y_pred,"y_true": y_true,"best_params": best_row.to_dict(),"metrics": final_df }

    print(f"Saved final model, metrics, and training curve for {model_class.__name__} at {final_out}")